# Stage 1: CyTOF preprocessing

This notebook runs raw data loading, CD45/GFP filtering, and GFP splitting from the intended checkout. It reads the authoritative annotated raw CSV directory and writes to the selected run directory.

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from either the checkout root or its notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
# Prefer this checkout over another installed copy or worktree of app.
sys.path.insert(0, str(PROJECT_ROOT))

import app  # noqa: E402

print(f"Kernel executable: {sys.executable}")
print(f"Working directory: {Path.cwd().resolve()}")
print(f"Imported app package: {Path(app.__file__).resolve()}")

# Fail early if Python resolved app from outside this checkout.
if Path(app.__file__).resolve().parent != PROJECT_ROOT / "app":
    raise RuntimeError("app was imported from outside the intended checkout")

In [ ]:
import pandas as pd
from IPython.display import display

from app.config import ANNOTATION_FILE, DEMO_MODE, OUTPUT_ROOT, active_data_dir
from app.config.config import (
    CD45_CUTOFFS_ASINH,
    EXPERIMENTS,
    GFP_CUTOFFS_ASINH,
)
from app.cell_selection import (
    add_cell_uid,
    attach_annotation_labels,
    load_raw_data,
    plot_cd45_histograms,
    plot_gfp_histograms,
    save_gfp_split,
    save_loading_summary,
    split_cells_by_gfp,
)

In [ ]:
INPUT_DIR = active_data_dir
# Input and shared filtering settings come from app.config.

In [ ]:
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Raw data folder not found: {INPUT_DIR}")
print(f"Demo mode: {DEMO_MODE}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_ROOT}")

## 1. Raw data loading

In [ ]:
DF_loaded = load_raw_data(INPUT_DIR, EXPERIMENTS)
DF_loaded = add_cell_uid(DF_loaded.copy())
DF_loaded = attach_annotation_labels(DF_loaded.copy(), ANNOTATION_FILE)
loading_summary = save_loading_summary(DF_loaded.copy(), OUTPUT_ROOT / "1. raw data")
print(f"Loaded cells: {len(DF_loaded)}")
display(loading_summary)

## 2. CD45 and GFP filtering

In [ ]:
# Plotting diagnostics only.
PLOT_ROOT = OUTPUT_ROOT / "2. CD45 + GFP gating plots"
plot_cd45_histograms(INPUT_DIR, EXPERIMENTS, PLOT_ROOT / "CD45 histograms", CD45_CUTOFFS_ASINH)
gfp_histogram_summary = plot_gfp_histograms(
    DF_loaded.copy(), EXPERIMENTS, GFP_CUTOFFS_ASINH, PLOT_ROOT / "GFP histograms"
)
gfp_histogram_summary.to_csv(PLOT_ROOT / "GFP_histogram_summary.csv", index=False)
display(gfp_histogram_summary)

### Apply GFP split

In [ ]:
# Apply and save the filtered groups.
gfp_splits, gfp_split_summary = split_cells_by_gfp(
    DF_loaded.copy(), EXPERIMENTS, GFP_CUTOFFS_ASINH, cd45_cutoff=2.0
)
split_files = save_gfp_split(gfp_splits, OUTPUT_ROOT / "3. GFP split after CD45 exclusion")
# gfp_split_summary.to_csv(
#     OUTPUT_ROOT / "3. GFP split after CD45 exclusion" / "GFP_split_summary.csv", index=False
# )
print(f"Refreshed split files: {len(split_files)}")
display(gfp_split_summary)